In [125]:
import pandas as pd

df_fbref = pd.read_csv("/content/fbref_big5_brasileirao_2526.csv")

print("shape: ")
print(df_fbref.shape)
print("head: ")
print(df_fbref.head())

print("------")
df_fbref.isnull().sum()

shape: 
(2953, 66)
head: 
   Rk               Player  Nation    Pos              Squad  \
0   1     Brenden Aaronson   usUSA  MF,FW       Leeds United   
1   2          Zach Abbott  engENG     DF  Nottingham Forest   
2   3  Jones El-Abdellaoui   maMAR  MF,FW         Celta Vigo   
3   4        Himad Abdelli   dzALG     MF          Marseille   
4   5        Himad Abdelli   dzALG     MF             Angers   

                Comp     Age    Born  MP  Starts   Min   90s  Gls  Ast  G+A  \
0  engPremier League  25-204  2000.0  35      28  2301  25.6    4    5    9   
1  engPremier League  20-001  2006.0   3       2   164   1.8    0    0    0   
2          esLa Liga  20-122  2006.0  22       4   681   7.6    2    0    2   
3          frLigue 1  26-178  1999.0   8       1   138   1.5    0    0    0   
4          frLigue 1  26-178  1999.0  13      11   943  10.5    2    0    2   

   G-PK  PK  PKatt  CrdY  CrdR  G+A-PK  Matches  Cmp  Att  Cmp%  TotDist  \
0     4   0      0     2     0    0.35

,0
Rk,0
Player,0
Nation,3
Pos,0
Squad,0
...,...
CS%,2744
PKA,2744
PKsv,2744
PKm,2744


In [126]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def converter_valor(valor):

    valor = valor.replace("€", "").strip()

    if "mi." in valor.lower():
        numero = float(valor.lower().replace("mi.", "").strip())
        return numero * 1_000_000

    if "mil" in valor.lower():
        numero = float(valor.lower().replace("mil", "").strip())
        return numero * 1_000

    return None

headers = {
    "User-Agent": "Mozilla/5.0"
}

jogadores =  []

for pagina in range(1, 9):
  print(f"Coletando página {pagina}")

  url = f"https://www.transfermarkt.com.br/spieler-statistik/wertvollstespieler/marktwertetop/plus/0?galerie=0&page={pagina}"
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "html.parser")

  for linha in soup.find_all("tr", class_=["odd", "even"]):
    nome = linha.find("td", class_="hauptlink")
    valor = linha.find("td", class_="rechts hauptlink")

    if nome and valor:
      jogador_obj = {
          "nome": nome.get_text(strip=True),
          "valor_mercado": converter_valor(valor.get_text(strip=True))
      }

      jogadores.append(jogador_obj)

  time.sleep(2)

df_transfer = pd.DataFrame(jogadores)
print(df_transfer.head())
print(df_transfer.shape)

df_transfer.to_csv(
    "transfermarkt_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

#Foram 200 jogadores do transfermarket e 2953 do fbref


Coletando página 1
Coletando página 2
Coletando página 3
Coletando página 4
Coletando página 5
Coletando página 6
Coletando página 7
Coletando página 8
             nome  valor_mercado
0    Lamine Yamal    200000000.0
1  Erling Haaland    200000000.0
2   Kylian Mbappé    180000000.0
3           Pedri    150000000.0
4   Michael Olise    150000000.0
(200, 2)


In [127]:
import unicodedata

def normalizar_nome(nome):
  nome = str(nome)

  nome = unicodedata.normalize(
      "NFKD", nome
  ).encode(
      "ascii", "ignore"
  ).decode(
      "utf-8"
  )

  nome = nome.lower().strip()

  return nome



In [128]:
#teste de identificação de registros inconsistentes em df_fbref

df_fbref.groupby(
    ["Player", "Born", "Squad"]
).size().value_counts()

chave_jogador = [
    "Player",
    "Born",
    "Squad"
]

duplicados = (
    df_fbref
    .groupby(chave_jogador)
    .size()
    .reset_index(name="qtd")
)

duplicados[duplicados["qtd"] > 1]

,Player,Born,Squad,qtd
445,Carlos Eduardo,1989.0,Mirassol,8
446,Carlos Eduardo,1996.0,Mirassol,8
1883,Matheusinho,1997.0,Vitória,8
1884,Matheusinho,1998.0,Vitória,8


In [129]:
#pegando apenas o primeiro registro quando existe mais de 1 com o mesmo nome, nascimento e time

df_fbref_limpo = (
    df_fbref
    .drop_duplicates(
        subset=["Player", "Born", "Squad"],
        keep="first"
    )
    .copy()
)


print(df_fbref.shape)
print(df_fbref_limpo.shape)

(2953, 66)
(2925, 66)


In [130]:
df_fbref_limpo[
    df_fbref_limpo["Nation"].isna()
][["Player", "Squad", "Age", "Born", "Nation"]]

df_fbref_limpo[
    df_fbref_limpo["Nation"].isna()
][["Player", "Squad", "Age", "Born", "Nation"]]


,Player,Squad,Age,Born,Nation
1459,Nathan Mbala,Metz,18-084,2008.0,NaN
1714,Luis Orejuela,Mallorca,NaN,NaN,NaN
2265,Yael Trepy,Cagliari,NaN,NaN,NaN


In [131]:
df_fbref_limpo["nome_norm"] = (
    df_fbref_limpo["Player"]
    .apply(normalizar_nome)
)

df_transfer["nome_norm"] = (
    df_transfer["nome"]
    .apply(normalizar_nome)
)

print(df_fbref_limpo[
    ["Player", "nome_norm", "Born"]
].head())

print(df_transfer.head())

                Player            nome_norm    Born
0     Brenden Aaronson     brenden aaronson  2000.0
1          Zach Abbott          zach abbott  2006.0
2  Jones El-Abdellaoui  jones el-abdellaoui  2006.0
3        Himad Abdelli        himad abdelli  1999.0
4        Himad Abdelli        himad abdelli  1999.0
             nome  valor_mercado       nome_norm
0    Lamine Yamal    200000000.0    lamine yamal
1  Erling Haaland    200000000.0  erling haaland
2   Kylian Mbappé    180000000.0   kylian mbappe
3           Pedri    150000000.0           pedri
4   Michael Olise    150000000.0   michael olise


In [144]:
df_pre_final = df_fbref_limpo.merge(
    df_transfer[
        ["nome_norm", "valor_mercado"]
    ],
    on="nome_norm",
    how="left"
)

df_pre_final["idade"] = (
    df_pre_final["Age"]
    .str.split("-")
    .str[0]
)

df_pre_final = df_pre_final.rename(
    columns={
        "nome_norm": "nome",
        "Pos": "posicao",
        "Nation": "nacionalidade",
        "Squad": "clube",
        "MP": "jogos",
        "Min": "minutos",
        "Gls": "gols",
        "Ast": "assistencias",
        "CrdY": "amarelos",
        "CrdR": "vermelhos",
        "Att": "passes_tent",
        "Cmp%": "pct_passes",
        "PrgDist": "passes_prog",
        "SoT": "chutes_gol",
        "TklW": "desarmes",
        "Int": "interceptacoes",
        "Blocks": "bloqueios",
        "Clr": "cortes",
        "Saves": "defesas_gk",
        "Save%": "pct_defesas",
        "CS": "clean_sheets",
        "Sh": "chutes",
        "A-xAG": "xG"
    }
)

colunas_zerar = [
    "gols",
    "assistencias",
    "amarelos",
    "vermelhos",
    "passes_tent",
    "pct_passes",
    "passes_prog",
    "chutes",
    "chutes_gol",
    "xG",
    "desarmes",
    "interceptacoes",
    "bloqueios",
    "cortes",
    "defesas_gk",
    "pct_defesas",
    "clean_sheets"
]

colunas_finais = [
    "nome",
    "valor_mercado",
    "posicao",
    "idade",
    "nacionalidade",
    "clube",
    "jogos",
    "minutos",
    "gols",
    "assistencias",
    "amarelos",
    "vermelhos",
    "passes_tent",
    "pct_passes",
    "passes_prog",
    "chutes",
    "chutes_gol",
    "xG",
    "desarmes",
    "interceptacoes",
    "bloqueios",
    "cortes",
    "defesas_gk",
    "pct_defesas",
    "clean_sheets"
]

df_pre_final[colunas_zerar] = (
    df_pre_final[colunas_zerar]
    .fillna(0)
)

dataset_final = df_pre_final[colunas_finais]

# organizando por valor de mercado
dataset_sorted = dataset_final[
    dataset_final["valor_mercado"].notna()
].sort_values(
    by="valor_mercado",
    ascending=False
)

print(dataset_sorted.head(20))

print(dataset_final.head())

print("-------------------------------")

print(
    "Jogadores com valor de mercado:",
    dataset_final["valor_mercado"]
    .notna()
    .sum()
)

print(
    "Jogadores sem valor de mercado:",
    dataset_final["valor_mercado"]
    .isna()
    .sum()
)

dataset_sorted.to_csv(
    "dataset_sorted.csv",
    index=False,
    encoding="utf-8-sig"
)


dataset_final.to_csv(
    "dataset_final.csv",
    index=False,
    encoding="utf-8-sig"
)

                       nome  valor_mercado posicao idade nacionalidade  \
950          erling haaland    200000000.0      FW    25         noNOR   
2403           lamine yamal    200000000.0   MF,FW    18         esESP   
1462          kylian mbappe    180000000.0      FW    27         frFRA   
1776                  pedri    150000000.0      MF    23         esESP   
1698          michael olise    150000000.0      MF    24         frFRA   
1110        vinicius junior    140000000.0   FW,MF    25         brBRA   
2350                vitinha    140000000.0      MF    26         ptPOR   
1635             joao neves    140000000.0      MF    21         ptPOR   
1229  khvicha kvaratskhelia    140000000.0      FW    25         geGEO   
2351                vitinha    140000000.0   FW,MF    26         ptPOR   
235         jude bellingham    130000000.0      MF    22        engENG   
1902            declan rice    120000000.0      MF    27        engENG   
659             desire doue    1200000